## Our Results

In [54]:
import os
import re
import ast
import pandas as pd

models = ['sasrec', 'caser', 'gru']
datasets = ["rc15_results", "retail_rocket_results"]
# MODIFY PATH TO div4rec dir of Paparellas
div4rec_path = "/home/marek/Kinit/my_smorl/data"
results_dir = "/home/marek/Kinit/my_smorl/Plots/Havrila_both"
os.makedirs(results_dir, exist_ok=True)


#### Plot weights and alphas comparison: FIX vs UNFIX

In [115]:
import pandas as pd
from matplotlib import pyplot as plt

def plot_helper(ax, dfv, dfs, PLOT, window=1, dataline = 'metrics'):
    ax.set_ylabel(PLOT, color='black')
    if dataline == 'metrics':
        ax.plot(dfv['steps'], dfv[PLOT].rolling(window).mean(), color='black', linewidth=2, label='vanilla')
    ax.plot(dfs[0]['steps'], dfs[0][PLOT].rolling(window).mean(), color='gray', linewidth=2, label='rl_zeroed')
    # fixed
    ax.plot(dfs[1]['steps'], dfs[1][PLOT].rolling(window).mean(), color='blue', linestyle=":", linewidth=2, label='fix_1_100')
    ax.plot(dfs[2]['steps'], dfs[2][PLOT].rolling(window).mean(), color='red', linestyle=":",linewidth=2, label='fix_1_111')
    ax.plot(dfs[3]['steps'], dfs[3][PLOT].rolling(window).mean(), color='green', linestyle=":", linewidth=2, label='fix_3_100')
    ax.plot(dfs[4]['steps'], dfs[4][PLOT].rolling(window).mean(), color='maroon', linestyle=":", linewidth=2, label='fix_3_111')
    ax.plot(dfs[5]['steps'], dfs[5][PLOT].rolling(window).mean(), color='orange', linestyle=":", linewidth=2, label='fix_10_100')
    ax.plot(dfs[6]['steps'], dfs[6][PLOT].rolling(window).mean(), color='yellow', linestyle=":", linewidth=2, label='fix_10_111')
    
    ax.plot(dfs[7]['steps'], dfs[7][PLOT].rolling(window).mean(), color='blue', linewidth=1, label='unfix_1_100')
    ax.plot(dfs[8]['steps'], dfs[8][PLOT].rolling(window).mean(), color='red', linewidth=1, label='unfix_1_111')
    ax.plot(dfs[9]['steps'], dfs[9][PLOT].rolling(window).mean(), color='green', linewidth=1, label='unfix_3_100')
    ax.plot(dfs[10]['steps'], dfs[10][PLOT].rolling(window).mean(), color='maroon', linewidth=1, label='unfix_3_111')
    ax.plot(dfs[11]['steps'], dfs[11][PLOT].rolling(window).mean(), color='orange', linewidth=1, label='unfix_10_100')
    ax.plot(dfs[12]['steps'], dfs[12][PLOT].rolling(window).mean(), color='yellow', linewidth=1, label='unfix_10_111')

def plot_metrics(basepath, dataset, model, replica, testorval, window=1):
    fixpath = f"{basepath}/Havrila_fix/{dataset}"
    unfixpath = f"{basepath}/Havrila_unfix/{dataset}"
    vanilapath = f"{basepath}/Havrila_both/{dataset}"
    #datapath = f"/home/marek/Kinit/data/{dataset}"
    dataline = "metrics"
    dfv = pd.read_pickle(f"{vanilapath}/{model}/base_{dataline}")                     # common
    try:
        df0 = pd.read_pickle(f"{fixpath}/{model}/rl_1_000_{replica}_{dataline}")        # common
        df1 = pd.read_pickle(f"{fixpath}/{model}/rl_1_100_{replica}_{dataline}")        # fix
        df2 = pd.read_pickle(f"{fixpath}/{model}/rl_1_111_{replica}_{dataline}")        # fix
        df7 = pd.read_pickle(f"{unfixpath}/{model}/rl_1_100_{replica}_{dataline}")       # unfix
        if dataset == "rc15_results":
            df8 = pd.read_pickle(f"{vanilapath}/{model}/rl_111_{replica}_{dataline}")   # unfix
        else:
            df8 = pd.read_pickle(f"{unfixpath}/{model}/rl_1_111_{replica}_{dataline}")       # unfix     
        df3 = pd.read_pickle(f"{fixpath}/{model}/rl_3_100_{replica}_{dataline}")         # fix
        df4 = pd.read_pickle(f"{fixpath}/{model}/rl_3_111_{replica}_{dataline}")        # fix
        df9 = pd.read_pickle(f"{unfixpath}/{model}/rl_3_100_{replica}_{dataline}")      # unfix
        df10 = pd.read_pickle(f"{unfixpath}/{model}/rl_3_111_{replica}_{dataline}")      # unfix
        df5 = pd.read_pickle(f"{fixpath}/{model}/rl_10_100_{replica}_{dataline}")       # fix
        df6 = pd.read_pickle(f"{fixpath}/{model}/rl_10_111_{replica}_{dataline}")       # fix
        df11 = pd.read_pickle(f"{unfixpath}/{model}/rl_10_100_{replica}_{dataline}")    # unfix
        df12 = pd.read_pickle(f"{unfixpath}/{model}/rl_10_111_{replica}_{dataline}")    # unfix
        #df8 = pd.read_pickle(f"{filepath}/{model}/rl_075_100_{replica}_{dataline}")    # skip
        #df9 = pd.read_pickle(f"{filepath}/{model}/rl_075_111_{replica}_{dataline}")    # skip
        dfs = [df0, df1, df2, df3, df4, df5, df6, df7, df8, df9, df10, df11, df12]
    except:
        print("Error during loading dataframes")
        dfs = []
    
    #~~~~~~ Plot cov values ~~~~~~
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'Havrila : {dataset}-{model}-{replica}-{testorval}', fontsize=12, y=0.94)
     
    PLOT = f"cov_{testorval}_10"
    ax1 = axs[0, 0]
    ax1.set_title("COV 10", fontsize=10, pad=10)
    #ax1.set_ylim(0.1, 0.75)
    ax1.set_ylabel(PLOT, color="black")
    plot_helper(ax1, dfv, dfs, PLOT)
         
    #~~~~~~ Plot hr ~~~~~~   
    PLOT = f"hr_{testorval}_10"
    ax5 = axs[0, 1]
    ax5.set_title("HR 10", fontsize=10, pad=10)
    #ax5.set_ylim(0.3, 0.55)
    plot_helper(ax5, dfv, dfs, PLOT)
        
    # ~~~~~~ Plot losses ~~~~~~
    dataline = "loss"
    dflv = pd.read_pickle(f"{vanilapath}/{model}/base_{dataline}")                     # common
    try:
        dfl0 = pd.read_pickle(f"{fixpath}/{model}/rl_1_000_{replica}_{dataline}")        # common
        dfl1 = pd.read_pickle(f"{fixpath}/{model}/rl_1_100_{replica}_{dataline}")        # fix
        dfl2 = pd.read_pickle(f"{fixpath}/{model}/rl_1_111_{replica}_{dataline}")        # fix
        dfl7 = pd.read_pickle(f"{unfixpath}/{model}/rl_1_100_{replica}_{dataline}")       # unfix 
        if dataset == "rc15_results":
            dfl8 = pd.read_pickle(f"{vanilapath}/{model}/rl_111_{replica}_{dataline}")       # unfix 
        else:
            dfl8 = pd.read_pickle(f"{unfixpath}/{model}/rl_1_111_{replica}_{dataline}")
        dfl3 = pd.read_pickle(f"{fixpath}/{model}/rl_3_100_{replica}_{dataline}")         # fix
        dfl4 = pd.read_pickle(f"{fixpath}/{model}/rl_3_111_{replica}_{dataline}")        # fix
        dfl9 = pd.read_pickle(f"{unfixpath}/{model}/rl_3_100_{replica}_{dataline}")      # unfix
        dfl10 = pd.read_pickle(f"{unfixpath}/{model}/rl_3_111_{replica}_{dataline}")      # unfix
        dfl5 = pd.read_pickle(f"{fixpath}/{model}/rl_10_100_{replica}_{dataline}")       # fix
        dfl6 = pd.read_pickle(f"{fixpath}/{model}/rl_10_111_{replica}_{dataline}")       # fix
        dfl11 = pd.read_pickle(f"{unfixpath}/{model}/rl_10_100_{replica}_{dataline}")    # unfix
        dfl12 = pd.read_pickle(f"{unfixpath}/{model}/rl_10_111_{replica}_{dataline}")    # unfix
        #df8 = pd.read_pickle(f"{filepath}/{model}/rl_075_100_{replica}_{dataline}")    # skip
        #df9 = pd.read_pickle(f"{filepath}/{model}/rl_075_111_{replica}_{dataline}")    # skip
        dfls = [dfl0, dfl1, dfl2, dfl3, dfl4, dfl5, dfl6, dfl7, dfl8, dfl9, dfl10, dfl11, dfl12]
    except:
        print("ERROR during loading dataframes")
        dfls = []

    PLOT = "loss"
    ax8 = axs[1, 0]
    ax8.set_title("Loss", fontsize=10, pad=10)
    ax8.yaxis.set_label_position("left")
    ax8.set_ylim(0, 10)
    ax8.yaxis.tick_left()
    plot_helper(ax8, dflv, dfls, PLOT, window=5, dataline='loss')
          
    PLOT = "plain"
    ax9 = axs[1, 1]
    ax9.set_title("Loss components", fontsize=10, pad=10)
    ax9.set_ylim(0, 10)
    plot_helper(ax9, dflv, dfls, PLOT, window=5, dataline='loss')

    PLOT = "smorl"
    ax10 = ax9.twinx()
    ax10.set_ylabel(PLOT, color='green')
    ax10.set_ylim(0, 10)
    plot_helper(ax10, dfl0, dfls, PLOT, window=5, dataline='loss')
    
    
    for i, ax in enumerate(axs.flat):
        ax.yaxis.grid(True, color='lightgray', linewidth=0.5)
        ax.tick_params(axis='y', labelsize=8)
        for child in ax.figure.axes:
            child.tick_params(axis='y', labelsize=8)
    
    handles, labels = ax1.get_legend_handles_labels()
    seen = set()
    unique = [(h, l) for h, l in zip(handles, labels) if not (l in seen or seen.add(l))]
    fig.legend(*zip(*unique), loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.02), fontsize='medium')
    
    fig.savefig(f"{basepath}VARIANTS01-{model}-{dataset}-{replica}-{testorval}.png", dpi=300, bbox_inches='tight')
    plt.close(fig)

models = ["sasrec", "caser", "gru"]
datasets = ["retail_rocket_results", "rc15_results"]
basepath = "/home/marek/Kinit/my_smorl/Plots/"
for dataset in datasets:
    for model in models: 
        for replica in ['main', 'target']:
            for variant in ['val']:
                plot_metrics(basepath, dataset, model, replica, variant)
            

#### Plot weights for a model

In [6]:
import os
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib.cm as cm
import scienceplots
plt.style.use(['science', 'no-latex'])

grey = cm.get_cmap('Greys')
green = cm.get_cmap('Greens')

def plot_helper(ax, df0, dfs, df6, PLOT, col1, col2, window=1):
    yname = ylab_helper(PLOT)
    ax.set_ylabel(yname, color=col1)
    ax.plot(df0['steps'], df0[PLOT].rolling(window).mean(), color=col1, linestyle=":", linewidth=2)
    ax.plot(dfs[0]['steps'], dfs[0][PLOT].rolling(window).mean(), color=col1, linestyle="--", linewidth=1)
    if len(dfs) > 0:
        ax.plot(dfs[1]['steps'], dfs[1][PLOT].rolling(window).mean(), color=col2(0.2), linewidth=1)
        ax.plot(dfs[2]['steps'], dfs[2][PLOT].rolling(window).mean(), color=col2(0.3), linewidth=1)
        ax.plot(dfs[3]['steps'], dfs[3][PLOT].rolling(window).mean(), color=col2(0.4), linewidth=1)
        ax.plot(dfs[4]['steps'], dfs[4][PLOT].rolling(window).mean(), color=col2(0.5), linewidth=1)
        ax.plot(dfs[5]['steps'], dfs[5][PLOT].rolling(window).mean(), color=col2(0.6), linewidth=1)
    ax.plot(df6['steps'], df6[PLOT].rolling(window).mean(), color=col1, linewidth=1)
    
def ylab_helper(name):
    if name.startswith("hr_") and name.endswith("_10"):
        label = 'HR@10'
    elif name.startswith("ndcg_") and name.endswith("_10"):
        label = 'nDCG@10'
    elif name.startswith("cov_") and name.endswith("_10"):
        label = 'COV@10'
    elif name.startswith("nov_") and name.endswith("_10"):
        label = 'NOV@10'
    elif name.startswith("rep_") and name.endswith("_5"):
        label = 'REP@5'
    elif name == "smorl":
        label = 'RL-head loss'
    elif name == "plain":
        label = 'self-supervised loss'
    else:
        label = name
    return label
    
def plot_metrics(basepath, dataset, model, replica, testorval, window=1):
    filepath = f"{basepath}/{dataset}/{model}"
    vanillapath = f"/home/marek/Kinit/my_smorl/Plots/Havrila_both/{dataset}/{model}"
    dataline = "metrics"
    df0 = pd.read_pickle(f"{vanillapath}/base_{dataline}")
    try:
        dfa = pd.read_pickle(f"{filepath}/rl_1_100_{replica}_{dataline}")
        df1 = pd.read_pickle(f"{filepath}/rl_1_001_{replica}_{dataline}")
        df2 = pd.read_pickle(f"{filepath}/rl_1_010_{replica}_{dataline}")
        df3 = pd.read_pickle(f"{filepath}/rl_1_011_{replica}_{dataline}")
        df4 = pd.read_pickle(f"{filepath}/rl_1_110_{replica}_{dataline}")
        df5 = pd.read_pickle(f"{filepath}/rl_1_101_{replica}_{dataline}")
        dfs = [dfa, df1, df2, df3, df4, df5]
    except:
        dfs = []
    df6 = pd.read_pickle(f"{filepath}/rl_1_111_{replica}_{dataline}")
    
    #~~~~~~ Plot cov + nov values ~~~~~~
    fig, axs = plt.subplots(3, 2, figsize=(10, 10))
    fig.suptitle(f'Havrila : {dataset}-{model}-{replica}-{testorval}', fontsize=12, y=0.94)
     
    PLOT = f"cov_{testorval}_10"
    ax1 = axs[0, 0]
    #ax1.set_title("COV 10 / NOV 10", fontsize=10, pad=10)
    ax1.set_ylim(0.1, 0.75)
    yname = ylab_helper(PLOT)
    ax1.set_ylabel(yname, color="black")
    ax1.plot(df0['steps'], df0[PLOT].rolling(window).mean(), color='black', linestyle=":", linewidth=2, label="vanilla")
    ax1.plot(dfs[0]['steps'], dfs[0][PLOT].rolling(window).mean(), color='black', linestyle="--", linewidth=1, label="SQN")  
    if len(dfs) > 0:
        ax1.plot(dfs[1]['steps'], dfs[1][PLOT].rolling(window).mean(), color='0.2', linewidth=1)
        ax1.plot(dfs[2]['steps'], dfs[2][PLOT].rolling(window).mean(), color='0.3', linewidth=1)
        ax1.plot(dfs[3]['steps'], dfs[3][PLOT].rolling(window).mean(), color='0.4', linewidth=1)
        ax1.plot(dfs[4]['steps'], dfs[4][PLOT].rolling(window).mean(), color='0.5', linewidth=1)
        ax1.plot(dfs[5]['steps'], dfs[5][PLOT].rolling(window).mean(), color='0.6', linewidth=1)
    ax1.plot(df6['steps'], df6[PLOT].rolling(window).mean(), color="black", linewidth=1, label="RL")
    
    PLOT = f"nov_{testorval}_10"
    ax2 = ax1.twinx()
    ax2.set_ylim(0.1, 0.75)
    plot_helper(ax2, df0, dfs, df6, PLOT, 'green', green)
       
    #~~~~~~ Plot cov + nov aligned ~~~~~~
    PLOT = f"cov_{testorval}_10"
    ax3 = axs[0, 1]
    #ax3.set_title("COV 10 / NOV 10 Alignment", fontsize=10, pad=10)
    yname = ylab_helper(PLOT)
    ax3.set_ylabel(yname, color='black')
    ax3.plot(df0['steps'], df0[PLOT].rolling(window).mean(), color='black', linestyle=":", linewidth=2)
    ax3.plot(dfs[0]['steps'], dfs[0][PLOT].rolling(window).mean(), color='black', linestyle=":", linewidth=2)
    if len(dfs) > 0:
        ax3.plot(dfs[1]['steps'], dfs[1][PLOT].rolling(window).mean(), color='0.2', linewidth=1)
        ax3.plot(dfs[2]['steps'], dfs[2][PLOT].rolling(window).mean(), color='0.3', linewidth=1)
        ax3.plot(dfs[3]['steps'], dfs[3][PLOT].rolling(window).mean(), color='0.4', linewidth=1)
        ax3.plot(dfs[4]['steps'], dfs[4][PLOT].rolling(window).mean(), color='0.5', linewidth=1)
        ax3.plot(dfs[5]['steps'], dfs[5][PLOT].rolling(window).mean(), color='0.6', linewidth=1)
    ax3.plot(df6['steps'], df6[PLOT].rolling(window).mean(), color='lightgray', linewidth=4)

    PLOT = f"nov_{testorval}_10"
    ax4 = ax3.twinx()
    plot_helper(ax4, df0, dfs, df6, PLOT, 'green', green)
    
    #~~~~~~ Plot hr + ndcg ~~~~~~   
    PLOT = f"hr_{testorval}_10"
    ax5 = axs[1, 0]
    #ax5.set_title("HR 10 / NDCG 10", fontsize=10, pad=10)
    plot_helper(ax5, df0, dfs, df6, PLOT, 'black', grey)
        
    PLOT = f"ndcg_{testorval}_10"
    ax6 = ax5.twinx()
    plot_helper(ax6, df0, dfs, df6, PLOT, 'green', green) 
        
    # ~~~~~~ Plot repetitivness ~~~~~~
    PLOT = f"rep_{testorval}_5"
    ax7 = axs[1, 1]
    #ax7.set_title("REP 5", fontsize=10, pad=10)
    ax7.yaxis.set_label_position("right")
    ax7.yaxis.tick_right()  
    plot_helper(ax7, df0, dfs, df6, PLOT, 'black', grey)
    
    # ~~~~~~ Plot losses ~~~~~~
    dataline = "loss"
    dfl0 = pd.read_pickle(f"{vanillapath}/base_{dataline}")
    try:
        dfla = pd.read_pickle(f"{filepath}/rl_1_100_{replica}_{dataline}")
        dfl1 = pd.read_pickle(f"{filepath}/rl_1_001_{replica}_{dataline}")
        dfl2 = pd.read_pickle(f"{filepath}/rl_1_010_{replica}_{dataline}")
        dfl3 = pd.read_pickle(f"{filepath}/rl_1_011_{replica}_{dataline}")
        dfl4 = pd.read_pickle(f"{filepath}/rl_1_110_{replica}_{dataline}")
        dfl5 = pd.read_pickle(f"{filepath}/rl_1_101_{replica}_{dataline}")
        dfls = [dfla, dfl1, dfl2, dfl3, dfl4, dfl5]
    except:
        dfls = []
    dfl6 = pd.read_pickle(f"{filepath}/rl_1_111_{replica}_{dataline}") 
    
    PLOT = "loss"
    ax8 = axs[2, 0]
    #ax8.set_title("Loss", fontsize=10, pad=10)
    ax8.yaxis.set_label_position("left")
    ax8.set_ylim(0, 10)
    ax8.set_xlabel("Steps", color='black')
    ax8.yaxis.tick_left()
    plot_helper(ax8, dfl0, dfls, dfl6, PLOT, 'black', grey, 5)
          
    PLOT = "plain"
    ax9 = axs[2, 1]
    #ax9.set_title("Loss components", fontsize=10, pad=10)
    ax9.set_ylim(0, 10)
    yname = ylab_helper(PLOT)
    ax9.set_ylabel(yname, color='black')
    ax9.set_xlabel("Steps", color='black')
    for frame in dfls[1:]:
        ax9.plot(frame['steps'], frame[PLOT].rolling(5).mean(), color='lightgray', linewidth=1)
    ax9.plot(dfl6['steps'], dfl6[PLOT].rolling(5).mean(), color='black', linewidth=1)
    ax9.plot(dfls[0]['steps'], dfls[0][PLOT].rolling(5).mean(), color='black', linestyle="--", linewidth=1)

    
    PLOT = "smorl"
    ax10 = ax9.twinx()
    yname = ylab_helper(PLOT)
    ax10.set_ylabel(yname, color='green')
    ax10.set_ylim(0, 10)
    for frame in dfls[1:]:
        ax10.plot(frame['steps'], frame[PLOT].rolling(5).mean(), color='lightgreen', linewidth=1)
    ax10.plot(dfls[0]['steps'], dfls[0][PLOT].rolling(5).mean(), linestyle="--", color='lightgreen', linewidth=1)
    
    
    
    for i, ax in enumerate(axs.flat):
        ax.yaxis.grid(True, color='lightgray', linewidth=0.5)
        ax.tick_params(axis='y', labelsize=8)
        ax.tick_params(axis='x', labelsize=8)
        for child in ax.figure.axes:
            child.tick_params(axis='y', labelsize=8)
            child.tick_params(axis='x', labelsize=8)
    
    fig.legend(
        loc='lower center',
        ncol=2,                  # number of columns in legend
        bbox_to_anchor=(0.5, -0.02),  # center below figure
        fontsize='medium'
    )
    plt.subplots_adjust(wspace=0.3, hspace=0.2)

    fig.savefig(f"{basepath}{dataset}-{model}-{replica}-{testorval}.png", dpi=300, bbox_inches='tight')
    plt.close(fig)

models = ["gru", "caser", "sasrec"]#, "nextitnet"]
datasets = ["rc15_results", "retail_rocket_results"]
#os.makedirs(results_dir, exist_ok=True)

basepath = "/home/marek/Kinit/my_smorl/Plots/Havrila_fix/"
for dataset in datasets:
    for model in models: 
        for replica in ['main', 'target']:
            for variant in ['val']:
                plot_metrics(basepath, dataset, model, replica, variant)

/tmp/ipykernel_2963774/4220504397.py:8: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  grey = cm.get_cmap('Greys')
/tmp/ipykernel_2963774/4220504397.py:9: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  green = cm.get_cmap('Greens')


#### Simple


In [8]:
import os
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib.cm as cm
import scienceplots
plt.style.use(['science', 'no-latex'])

grey = cm.get_cmap('Greys')
green = cm.get_cmap('Greens')

def plot_helper(ax, df0, dfs, df6, PLOT, col1, col2, window=1):
    yname = ylab_helper(PLOT)
    ax.set_ylabel(yname, color=col1)
    PLOT1 = PLOT
    if PLOT != 'smorl':
        if PLOT == 'plain':
            PLOT1 = 'loss'
        ax.plot(df0['steps'], df0[PLOT1].rolling(window).mean(), color=col1, linestyle=":", linewidth=2)
    ax.plot(dfs[0]['steps'], dfs[0][PLOT].rolling(window).mean(), color=col1, linestyle="--", linewidth=1)
    if len(dfs) > 0:
        ax.plot(dfs[1]['steps'], dfs[1][PLOT].rolling(window).mean(), color=col2(0.2), linewidth=1)
        ax.plot(dfs[2]['steps'], dfs[2][PLOT].rolling(window).mean(), color=col2(0.3), linewidth=1)
        ax.plot(dfs[3]['steps'], dfs[3][PLOT].rolling(window).mean(), color=col2(0.4), linewidth=1)
        ax.plot(dfs[4]['steps'], dfs[4][PLOT].rolling(window).mean(), color=col2(0.5), linewidth=1)
        ax.plot(dfs[5]['steps'], dfs[5][PLOT].rolling(window).mean(), color=col2(0.6), linewidth=1)
    ax.plot(df6['steps'], df6[PLOT].rolling(window).mean(), color=col1, linewidth=1)
    
def ylab_helper(name):
    if name.startswith("hr_") and name.endswith("_10"):
        label = 'HR@10'
    elif name.startswith("ndcg_") and name.endswith("_10"):
        label = 'nDCG@10'
    elif name.startswith("cov_") and name.endswith("_10"):
        label = 'COV@10'
    elif name.startswith("nov_") and name.endswith("_10"):
        label = 'NOV@10'
    elif name.startswith("rep_") and name.endswith("_5"):
        label = 'REP@5'
    elif name == "smorl":
        label = 'RL-head loss'
    elif name == "plain":
        label = 'self-supervised loss'
    else:
        label = name
    return label
    
def plot_metrics(basepath, dataset, model, replica, testorval, window=1):
    filepath = f"{basepath}/{dataset}/{model}"
    vanillapath = f"/home/marek/Kinit/my_smorl/Plots/Havrila_both/{dataset}/{model}"
    dataline = "metrics"
    df0 = pd.read_pickle(f"{vanillapath}/base_{dataline}")
    try:
        dfa = pd.read_pickle(f"{filepath}/rl_1_100_{replica}_{dataline}")
        df1 = pd.read_pickle(f"{filepath}/rl_1_001_{replica}_{dataline}")
        df2 = pd.read_pickle(f"{filepath}/rl_1_010_{replica}_{dataline}")
        df3 = pd.read_pickle(f"{filepath}/rl_1_011_{replica}_{dataline}")
        df4 = pd.read_pickle(f"{filepath}/rl_1_110_{replica}_{dataline}")
        df5 = pd.read_pickle(f"{filepath}/rl_1_101_{replica}_{dataline}")
        dfs = [dfa, df1, df2, df3, df4, df5]
    except:
        dfs = []
    df6 = pd.read_pickle(f"{filepath}/rl_1_111_{replica}_{dataline}")
    
    #~~~~~~ Plot cov + nov values ~~~~~~
    fig, axs = plt.subplots(3, 2, figsize=(10, 10))
    fig.suptitle(f'Havrila : {dataset}-{model}-{replica}-{testorval}', fontsize=12, y=0.94)
     
    PLOT = f"cov_{testorval}_10"
    ax1 = axs[0, 0]
    #ax1.set_title("COV 10 / NOV 10", fontsize=10, pad=10)
    ax1.set_ylim(0.1, 0.75)
    yname = ylab_helper(PLOT)
    ax1.set_ylabel(yname, color="black")
    ax1.plot(df0['steps'], df0[PLOT].rolling(window).mean(), color='black', linestyle=":", linewidth=2, label="Vanilla")
    ax1.plot(dfs[0]['steps'], dfs[0][PLOT].rolling(window).mean(), color='black', linestyle="--", linewidth=1, label="SMORL4RS[1.0.0] - SQN")  
    if len(dfs) > 0:
        ax1.plot(dfs[1]['steps'], dfs[1][PLOT].rolling(window).mean(), color='0.2', linewidth=1, label="SMORL4RS[0.0.1]")
        ax1.plot(dfs[2]['steps'], dfs[2][PLOT].rolling(window).mean(), color='0.3', linewidth=1, label="SMORL4RS[0.1.0]")
        ax1.plot(dfs[3]['steps'], dfs[3][PLOT].rolling(window).mean(), color='0.4', linewidth=1, label="SMORL4RS[0.1.1]")
        ax1.plot(dfs[4]['steps'], dfs[4][PLOT].rolling(window).mean(), color='0.5', linewidth=1, label="SMORL4RS[1.1.0]")
        ax1.plot(dfs[5]['steps'], dfs[5][PLOT].rolling(window).mean(), color='0.6', linewidth=1, label="SMORL4RS[1.0.1]")
    ax1.plot(df6['steps'], df6[PLOT].rolling(window).mean(), color="black", linewidth=1, label="SMORL4RS[1.1.1]")
    
    PLOT = f"nov_{testorval}_10"
    ax2 = axs[0, 1]
    ax2.set_ylim(0.1, 0.75)
    yname = ylab_helper(PLOT)
    ax2.set_ylabel(yname, color='black')
    ax2.yaxis.set_label_position("right")
    ax2.yaxis.tick_right() 
    plot_helper(ax2, df0, dfs, df6, PLOT, 'black', grey)
       
    
    #~~~~~~ Plot hr + ndcg ~~~~~~   
    PLOT = f"hr_{testorval}_10"
    ax5 = axs[1, 0]
    #ax5.set_title("HR 10 / NDCG 10", fontsize=10, pad=10)
    plot_helper(ax5, df0, dfs, df6, PLOT, 'black', grey)
        
    PLOT = f"ndcg_{testorval}_10"
    ax6 = axs[1,1]
    ax6.yaxis.set_label_position("right")
    ax6.yaxis.tick_right() 
    plot_helper(ax6, df0, dfs, df6, PLOT, 'black', grey) 
        
    # ~~~~~~ Plot losses ~~~~~~
    dataline = "loss"
    dfl0 = pd.read_pickle(f"{vanillapath}/base_{dataline}")
    try:
        dfla = pd.read_pickle(f"{filepath}/rl_1_100_{replica}_{dataline}")
        dfl1 = pd.read_pickle(f"{filepath}/rl_1_001_{replica}_{dataline}")
        dfl2 = pd.read_pickle(f"{filepath}/rl_1_010_{replica}_{dataline}")
        dfl3 = pd.read_pickle(f"{filepath}/rl_1_011_{replica}_{dataline}")
        dfl4 = pd.read_pickle(f"{filepath}/rl_1_110_{replica}_{dataline}")
        dfl5 = pd.read_pickle(f"{filepath}/rl_1_101_{replica}_{dataline}")
        dfls = [dfla, dfl1, dfl2, dfl3, dfl4, dfl5]
    except:
        dfls = []
    dfl6 = pd.read_pickle(f"{filepath}/rl_1_111_{replica}_{dataline}") 
          
    PLOT = "plain"
    ax9 = axs[2, 0]
    #ax9.set_title("Loss components", fontsize=10, pad=10)
    ax9.set_ylim(0, 10)
    yname = ylab_helper(PLOT)
    ax9.set_ylabel(yname, color='black')
    ax9.set_xlabel("Steps", color='black')
    plot_helper(ax9, dfl0, dfls, dfl6, PLOT, 'black', grey, window=5)

    PLOT = "smorl"
    ax10 = axs[2, 1]
    yname = ylab_helper(PLOT)
    ax10.set_ylabel(yname, color='black')
    ax10.set_xlabel("Steps", color='black')
    ax10.yaxis.set_label_position("right")
    ax10.yaxis.tick_right() 
    ax10.set_ylim(0, 10)
    plot_helper(ax10, dfl0, dfls, dfl6, PLOT, 'black', grey, window=5)
    
    
    
    for i, ax in enumerate(axs.flat):
        ax.yaxis.grid(True, color='lightgray', linewidth=0.5)
        ax.tick_params(axis='y', labelsize=8)
        ax.tick_params(axis='x', labelsize=8)
        for child in ax.figure.axes:
            child.tick_params(axis='y', labelsize=8)
            child.tick_params(axis='x', labelsize=8)
    
    fig.legend(
        loc='lower center',
        ncol=2,                  # number of columns in legend
        bbox_to_anchor=(0.5, -0.02),  # center below figure
        fontsize='medium'
    )
    plt.subplots_adjust(wspace=0.1, hspace=0.1)

    fig.savefig(f"{basepath}Simpe-{dataset}-{model}-{replica}-{testorval}.png", dpi=300, bbox_inches='tight')
    plt.close(fig)

models = ["gru", "caser", "sasrec"]#, "nextitnet"]
datasets = ["rc15_results", "retail_rocket_results"]
#os.makedirs(results_dir, exist_ok=True)

basepath = "/home/marek/Kinit/my_smorl/Plots/Havrila_fix/"
for dataset in datasets:
    for model in models: 
        for replica in ['main', 'target']:
            for variant in ['val']:
                plot_metrics(basepath, dataset, model, replica, variant)

/tmp/ipykernel_2963774/3132726283.py:8: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  grey = cm.get_cmap('Greys')
/tmp/ipykernel_2963774/3132726283.py:9: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  green = cm.get_cmap('Greens')
